In [1]:
import requests
import re
from collections import Counter
from typing import List

class SimpleRAG:
    def __init__(self, model_name="llama3:8b", base_url="http://localhost:11434"):
        """
        初始化簡單RAG系統
        
        Args:
            model_name: Ollama模型名稱
            base_url: Ollama服務URL
            
        """
        self.model_name = model_name
        self.base_url = base_url
        self.chat_url = f"{base_url}/api/chat"
        self.documents = []  # 知識庫文檔
        self.processed_docs = []  # 預處理後的文檔（用於計算相似度）
    
    def _preprocess_text(self, text: str) -> List[str]:
        """
        改進的文本預處理：更好地處理中文分詞
        """
        # 保留中文字符、英文字母、數字
        text = re.sub(r'[^\w\s\u4e00-\u9fff]', ' ', text)
        
        # 中文按字符分割，英文按詞分割
        words = []
        current_word = ""
        
        for char in text:
            if '\u4e00' <= char <= '\u9fff':  # 中文字符
                if current_word:  # 如果有未完成的英文詞
                    words.append(current_word.lower())
                    current_word = ""
                words.append(char)  # 中文字符直接加入
            elif char.isalnum():  # 英文字母或數字
                current_word += char
            else:  # 空格或其他分隔符
                if current_word:
                    words.append(current_word.lower())
                    current_word = ""
        
        if current_word:  # 處理最後一個詞
            words.append(current_word.lower())
        
        # 過濾掉空字符
        return [word for word in words if word.strip()]
    
    def _calculate_tf_idf_similarity(self, query_words: List[str], doc_words: List[str]) -> float:
        """
        改進的TF-IDF相似度計算
        """
        if not query_words or not doc_words:
            return 0.0
        
        # 轉換為集合以提高查找效率
        doc_word_set = set(doc_words)
        doc_word_count = Counter(doc_words)
        
        # 計算匹配的詞數和權重
        total_score = 0.0
        matched_words = 0
        
        for query_word in query_words:
            if query_word in doc_word_set:
                matched_words += 1
                # TF: 詞在文檔中的頻率
                tf = doc_word_count[query_word] / len(doc_words)
                # 簡化權重：越長的詞權重越高
                word_weight = len(query_word) if len(query_word) > 1 else 1
                total_score += tf * word_weight
        
        if matched_words == 0:
            return 0.0
        
        # 正規化：考慮匹配詞的比例
        match_ratio = matched_words / len(query_words)
        return total_score * match_ratio
    
    def _jaccard_similarity(self, query_words: List[str], doc_words: List[str]) -> float:
        """
        計算Jaccard相似度
        """
        set1 = set(query_words)
        set2 = set(doc_words)
        
        intersection = set1 & set2
        union = set1 | set2
        
        return len(intersection) / len(union) if union else 0.0
    
    def add_documents(self, documents: List[str]):
        """
        添加文檔到知識庫
        """
        self.documents = documents
        self.processed_docs = [self._preprocess_text(doc) for doc in documents]
        print(f"✅ 已添加 {len(documents)} 個文檔到知識庫")
        
        # 顯示預處理結果用於除錯
        print("📝 文檔預處理示例:")
        for i, (doc, processed) in enumerate(zip(documents[:5], self.processed_docs[:5])):
            print(f"文檔{i+1}: {doc[:30]}...")
            print(f"處理後: {processed[:10]}...")
    
    def find_most_relevant_document(self, query: str, method: str = "tfidf") -> tuple:
        """
        找到最相關的文檔，返回文檔內容和相似度分數
        
        Args:
            query: 查詢文本
            method: 相似度計算方法 ("tfidf" 或 "jaccard")
        
        Returns:
            (best_document, best_similarity)
        """
        if not self.documents:
            return "", 0.0
        
        query_words = self._preprocess_text(query)
        print(f"🔍 查詢預處理: {query_words}")
        
        similarities = []
        
        for i, doc_words in enumerate(self.processed_docs):
            if method == "tfidf":
                similarity = self._calculate_tf_idf_similarity(query_words, doc_words)
            else:  # jaccard
                similarity = self._jaccard_similarity(query_words, doc_words)
            similarities.append((i, similarity))
            print(f"文檔{i+1}相似度: {similarity:.4f}")
        
        # 找到最高相似度的文檔
        best_doc_idx, best_similarity = max(similarities, key=lambda x: x[1])
        
        print(f"🎯 最佳匹配: 文檔{best_doc_idx + 1} (相似度: {best_similarity:.4f})")
        
        return self.documents[best_doc_idx], best_similarity
    
    def rag_query(self, query: str, method: str = "tfidf", similarity_threshold: float = 0.01) -> str:
        """
        RAG查詢：總是調用LLM，但會根據檢索結果調整提示詞
        
        Args:
            query: 用戶查詢
            method: 檢索方法 ("tfidf" 或 "jaccard")
            similarity_threshold: 相似度閾值，超過此值才認為找到相關文檔
        """
        # 檢索相關文檔
        best_doc, best_similarity = self.find_most_relevant_document(query, method)
        
        # 根據相似度決定提示詞策略
        if best_similarity > similarity_threshold:
            # 找到相關文檔，使用RAG模式
            prompt = f"""請基於以下參考資料回答問題：

參考資料：
{best_doc}

問題：請以繁體中文回答，{query}

請根據參考資料回答問題。如果參考資料不完全涵蓋問題，可以結合你的知識進行補充說明。"""
            
            context_info = f"[使用RAG模式，相似度: {best_similarity:.4f}]"
        else:
            # 沒有找到相關文檔，使用一般模式但告知LLM
            prompt = f"""問題：請以繁體中文回答，{query}

注意：我在知識庫中沒有找到與此問題直接相關的資料，請基於你的一般知識回答這個問題。"""
            
            context_info = f"[一般回答模式，最高相似度: {best_similarity:.4f}]"
        
        print(f"📋 {context_info}")
        
        # 總是調用LLM
        try:
            response = requests.post(
                self.chat_url,
                json={
                    "model": self.model_name,
                    "messages": [{"role": "user", "content": prompt}],
                    "stream": False
                },
                headers={"Content-Type": "application/json"},
                timeout=3000
            )
            
            if response.status_code == 200:
                result = response.json()
                return result["message"]["content"]
            else:
                return f"錯誤: HTTP {response.status_code} - {response.text}"
                
        except Exception as e:
            return f"請求錯誤: {str(e)}"

# 使用範例
def main(model_name="llama3:8b"):
    # RAG系統
    print(f"🤖 Ollama {model_name} 簡單RAG系統")
    print("=" * 50)
    
    # 初始化RAG系統
    rag = SimpleRAG(model_name)
    
    # 更好的知識庫（關鍵詞更明確）
    documents = [
        "Python程式語言特點：Python是解釋型語言，語法簡潔清晰，適合初學者學習。Python在數據分析、機器學習、網站開發方面應用廣泛。Python支援多種程式設計範式。",    
        "機器學習基礎概念：機器學習讓電腦從數據中自動學習規律。主要分為監督學習、非監督學習、強化學習三大類型。監督學習需要標註數據，非監督學習不需要標籤。",
        "深度學習神經網路：深度學習使用多層神經網路模擬大腦神經元。在圖像辨識、自然語言處理、語音辨識等任務表現優異。常用框架包括TensorFlow和PyTorch。",
        "資料庫管理系統：資料庫用於儲存和管理大量結構化數據。SQL是查詢語言標準。MySQL和PostgreSQL是常見的關聯式資料庫。MongoDB是文件型NoSQL資料庫。",
        "雲端計算服務模式：雲端計算提供網路運算資源。IaaS提供基礎設施，PaaS提供開發平台，SaaS提供軟體服務。AWS、Azure、Google Cloud是主要雲端服務提供商。"
    ]
    
    # 添加文檔到知識庫
    rag.add_documents(documents)
    
    # 更好的測試查詢（包含明確關鍵詞）
    test_queries = [
        "Python有什麼特點？",           # 應該匹配Python文檔
        "機器學習有哪些類型？",         # 應該匹配機器學習文檔
        "深度學習用什麼框架？",         # 應該匹配深度學習文檔
        "SQL是什麼？",                 # 應該匹配資料庫文檔
        "雲端服務有哪些模式？",          # 應該匹配雲端計算文檔        
        "如何學習程式設計？",            # 沒有相關文檔
        "什麼是區塊鏈？",               # 沒有相關文檔
        "Cup Cake是什麼?"               # 沒有相關文檔
    ]
    
    print(f"\n🔬 開始測試 {len(test_queries)} 個查詢:")
    print("-" * 50)
    
    for i, query in enumerate(test_queries, 1):
        print(f"\n【測試 {i}】")
        print(f"❓ 問題: {query}")
        
        # 使用TF-IDF方法
        answer = rag.rag_query(query, method="tfidf")
        print(f"🤖 回答: {answer}...")
        
        print("-" * 50)

if __name__ == "__main__":
    main(model_name="gpt-oss:20b")

🤖 Ollama gpt-oss:20b 簡單RAG系統
✅ 已添加 5 個文檔到知識庫
📝 文檔預處理示例:
文檔1: Python程式語言特點：Python是解釋型語言，語法簡潔...
處理後: ['python', '程', '式', '語', '言', '特', '點', 'python', '是', '解']...
文檔2: 機器學習基礎概念：機器學習讓電腦從數據中自動學習規律。主要分...
處理後: ['機', '器', '學', '習', '基', '礎', '概', '念', '機', '器']...
文檔3: 深度學習神經網路：深度學習使用多層神經網路模擬大腦神經元。在...
處理後: ['深', '度', '學', '習', '神', '經', '網', '路', '深', '度']...
文檔4: 資料庫管理系統：資料庫用於儲存和管理大量結構化數據。SQL是...
處理後: ['資', '料', '庫', '管', '理', '系', '統', '資', '料', '庫']...
文檔5: 雲端計算服務模式：雲端計算提供網路運算資源。IaaS提供基礎...
處理後: ['雲', '端', '計', '算', '服', '務', '模', '式', '雲', '端']...

🔬 開始測試 8 個查詢:
--------------------------------------------------

【測試 1】
❓ 問題: Python有什麼特點？
🔍 查詢預處理: ['python', '有', '什', '麼', '特', '點']
文檔1相似度: 0.2241
文檔2相似度: 0.0000
文檔3相似度: 0.0000
文檔4相似度: 0.0000
文檔5相似度: 0.0000
🎯 最佳匹配: 文檔1 (相似度: 0.2241)
📋 [使用RAG模式，相似度: 0.2241]
🤖 回答: **Python 的主要特點（繁體中文說明）**

| 特點 | 具體說明 |
|------|----------|
| **解釋型語言** | 程式碼不需先編譯成機器碼，直接由 Python 解譯執行；易於快速測試與迭代。 |
| **語法簡潔、可讀性高** | 使用縮排決定程式區塊（無大括號），文字化程度高，可使初學者快速上手，也方便維護大型